# SynthPanel Tuning with SynthBench

This notebook shows how to use SynthBench to score and tune
[SynthPanel](https://github.com/DataViking-Tech/synthpanel) outputs.

SynthPanel generates synthetic survey responses from persona-conditioned LLMs.
SynthBench measures how closely those responses match real human data.
Together, they form a generate-then-score loop for tuning synthetic survey quality.

**You need**: An API key for your chosen LLM provider (e.g., `OPENROUTER_API_KEY`).

## 1. Install Both Tools

In [ ]:
!pip install synthbench synthpanel

## 2. Create a Simple Instrument

An instrument defines the survey questions. Here we create a minimal YAML
inline to keep things self-contained. In practice you'd load a full survey file.

In [ ]:
from pathlib import Path

instrument_yaml = """\
title: Mini Survey
questions:
  - id: q1
    text: "How concerned are you about climate change?"
    options:
      - Very concerned
      - Somewhat concerned
      - Not too concerned
      - Not at all concerned
  - id: q2
    text: "How much do you trust the federal government to do what is right?"
    options:
      - Always
      - Most of the time
      - Some of the time
      - Never
  - id: q3
    text: "Do you think economic conditions in this country are getting better or worse?"
    options:
      - Getting better
      - About the same
      - Getting worse
"""

Path("mini_instrument.yaml").write_text(instrument_yaml)
print("Instrument saved to mini_instrument.yaml")

## 3. Create Personas

Personas define the demographic profiles that SynthPanel will role-play.
Different personas should produce different answer distributions — that's
the whole point of synthetic panels.

In [ ]:
personas_yaml = """\
personas:
  - id: young_liberal
    description: "A 25-year-old liberal college graduate living in a major city."
    demographics:
      age: 25
      ideology: liberal
      education: college
      region: urban
  - id: older_conservative
    description: "A 62-year-old conservative retiree living in a rural area."
    demographics:
      age: 62
      ideology: conservative
      education: high_school
      region: rural
  - id: moderate_suburban
    description: "A 40-year-old moderate professional living in the suburbs."
    demographics:
      age: 40
      ideology: moderate
      education: college
      region: suburban
"""

Path("mini_personas.yaml").write_text(personas_yaml)
print("Personas saved to mini_personas.yaml")

## 4. Run SynthPanel

Generate synthetic responses using your instrument and personas.
This calls the LLM API — make sure your API key is set.

**Note**: If synthpanel's CLI interface differs from what's shown here,
check `synthpanel --help` for the current syntax.

In [ ]:
import os

# Set your API key if not already in the environment
# os.environ["OPENROUTER_API_KEY"] = "sk-or-..."

if os.environ.get("OPENROUTER_API_KEY"):
    !synthpanel run \
        --instrument mini_instrument.yaml \
        --personas mini_personas.yaml \
        --output ./sp_output
    print("\nSynthPanel output saved to ./sp_output/")
else:
    print("Set OPENROUTER_API_KEY to run SynthPanel.")
    print("Remaining cells show what the workflow looks like.")

## 5. Score the Output with SynthBench

Now we use SynthBench's synthpanel provider adapter to score the output.
This compares the synthetic responses against real human ground truth data.

In [ ]:
sp_output = Path("./sp_output")

if sp_output.exists() and list(sp_output.glob("*.json")):
    # Score using the synthpanel adapter
    !synthbench run \
        --provider synthpanel \
        --suite smoke \
        --samples 10 \
        --output ./sb_scores
else:
    print("No SynthPanel output found. Run cell 4 first.")

## 6. Try Different Temperatures

Temperature affects how deterministic the LLM's responses are.
Lower temperature = more consistent but potentially less diverse responses.
Higher temperature = more varied but potentially noisier.

Run SynthPanel at different temperatures and compare the SynthBench scores
to find the sweet spot.

In [ ]:
# This cell demonstrates the tuning loop concept.
# When synthpanel supports --temperature, you can run:
#
#   for temp in [0.3, 0.7, 1.0]:
#       !synthpanel run --instrument mini_instrument.yaml \
#           --personas mini_personas.yaml \
#           --temperature {temp} \
#           --output ./sp_temp_{temp}
#       !synthbench run --provider synthpanel \
#           --suite smoke --output ./sb_temp_{temp}
#
# Then compare the scores to find the optimal temperature.

print("Temperature tuning workflow:")
print("  1. Run synthpanel at temp=0.3, 0.7, 1.0")
print("  2. Score each output with synthbench")
print("  3. Compare SPS scores to find the sweet spot")
print()
print("Lower temp -> more deterministic, may miss distributional variance")
print("Higher temp -> more variance, may introduce noise")

## 7. Compare Scores

Use SynthBench's built-in comparison to see which configuration produced
the best results, complete with statistical significance testing.

In [ ]:
# Compare two result files side by side
sb_scores = Path("./sb_scores")
score_files = sorted(sb_scores.glob("*.json")) if sb_scores.exists() else []

if len(score_files) >= 2:
    !synthbench compare {str(score_files[0])} {str(score_files[-1])}
else:
    print("Need at least 2 result files to compare.")
    print("Run the tuning loop above to generate multiple results.")
    print()
    print("The compare command outputs:")
    print("  - Side-by-side metric comparison")
    print("  - Delta and percentage difference")
    print("  - Statistical significance (p-value from paired bootstrap)")